In [ ]:
# =============================================
# CELL 1: Setup (run once per session)
# =============================================
import sys, subprocess, os

# Kaggle/NVIDIA API keys (use Kaggle Secrets in production)
os.environ["KAGGLE_USERNAME"] = os.environ.get("KAGGLE_USERNAME", "")
os.environ["KAGGLE_KEY"] = os.environ.get("KAGGLE_KEY", "")
os.environ["NVIDIA_API_KEY"] = os.environ.get("NVIDIA_API_KEY", "")

# Install dependencies
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "qiskit>=1.0.0", "qiskit-aer>=0.14.0",
    "pynvml", "pyyaml", "pandas", "kaggle", "requests",
    "--quiet"
])

# Clone repo
if not os.path.exists("openqsim-ai"):
    subprocess.check_call(["git", "clone", "https://github.com/lekkalaharsha/opensim-ai"])
else:
    os.chdir("openqsim-ai")
    subprocess.check_call(["git", "pull", "origin", "main"])
    os.chdir("..")

sys.path.append("openqsim-ai")
print("âœ… Setup complete")

In [ ]:
import sys, subprocess, os, shutil, importlib.util, re

# ============================================================
# INSTALL DEPENDENCIES
# ============================================================
print("[1/5] Installing dependencies...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--upgrade",
    "qiskit", "qiskit-aer", "pynvml", "pyyaml",
    "pandas", "kaggle>=2.2.2", "requests", "--quiet"
])
print("      Done.")

# ============================================================
# CLONE REPOSITORY
# ============================================================
print("[2/5] Cloning repository...")
repo_url = "https://github.com/lekkalaharsha/opensim-ai"
repo_dir = "opensim-ai"

if os.path.exists(repo_dir):
    shutil.rmtree(repo_dir)

subprocess.check_call(["git", "clone", repo_url])
print("      Done.")

# ============================================================
# PATCH: Fix absolute imports in kaggle/ files
# ============================================================
print("[3/5] Patching imports...")

def patch_kaggle_imports(file_path):
    """Replace 'from kaggle.X import' with 'from X import' in kaggle/ files."""
    with open(file_path, "r") as f:
        content = f.read()
    
    # Fix: from kaggle.xxx import -> from xxx import
    content = re.sub(r'from kaggle\.(\w+) import', r'from \1 import', content)
    # Fix: import kaggle.xxx -> import xxx (for kaggle/* modules)
    content = re.sub(r'import kaggle\.(\w+)', r'import \1', content)
    
    with open(file_path, "w") as f:
        f.write(content)

# Patch all Python files in the kaggle/ directory
kaggle_dir = f"{repo_dir}/kaggle"
for py_file in os.listdir(kaggle_dir):
    if py_file.endswith(".py"):
        patch_kaggle_imports(os.path.join(kaggle_dir, py_file))

# Also patch benchmark/runner.py which imports from kaggle
runner_path = f"{repo_dir}/benchmark/runner.py"
if os.path.exists(runner_path):
    patch_kaggle_imports(runner_path)

print("      Done.")

# ============================================================
# ADD TO PATH AND VERIFY GPU
# ============================================================
sys.path.insert(0, repo_dir)
sys.path.insert(0, kaggle_dir)

# Verify GPU
from backend.environment import collect_environment_metadata
env = collect_environment_metadata()
print(f"      GPU: {env.gpu_name} ({env.gpu_memory_mb} MB)")
print(f"      Qiskit: {env.qiskit_version}")

# ============================================================
# IMPORT MODULES DIRECTLY
# ============================================================
def import_from_file(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

# Import from patched files
kaggle_env = import_from_file("kaggle_env", f"{kaggle_dir}/environment.py")
kaggle_ckpt = import_from_file("kaggle_ckpt", f"{kaggle_dir}/checkpoint.py")
kaggle_client = import_from_file("kaggle_client", f"{kaggle_dir}/api_client.py")
kaggle_runner = import_from_file("kaggle_runner", f"{kaggle_dir}/runner.py")
kaggle_assembler = import_from_file("kaggle_assembler", f"{kaggle_dir}/dataset_assembler.py")

KaggleRunner = kaggle_runner.KaggleRunner
DatasetAssembler = kaggle_assembler.DatasetAssembler
KaggleAPIClient = kaggle_client.KaggleAPIClient

print("      Modules loaded.")

# ============================================================
# RUN SWEEP
# ============================================================
print("[4/5] Running benchmark sweep...")

runner = KaggleRunner(
    sweep_config_path=f"{repo_dir}/benchmark/sweep_config_0a.yaml",
    output_dir="/kaggle/working/benchmark_outputs",
    checkpoint_interval=10,
    artifact_interval=50,
    kaggle_dataset="lekkalaharsha/openqsim-benchmarks",
)

result = runner.run()
print(f"      Done: {result.completed_count} records")

# ============================================================
# ASSEMBLE DATASET
# ============================================================
print("[5/5] Assembling dataset...")

assembler = DatasetAssembler(
    raw_dir="/kaggle/working/benchmark_outputs/raw",
    dataset_dir="/kaggle/working/benchmark_outputs/datasets/openqsim_v0.1-small",
)
manifest = assembler.assemble()

# ============================================================
# FINAL PUSH
# ============================================================
client = KaggleAPIClient(dataset="lekkalaharsha/openqsim-benchmarks")
client.push_benchmark_outputs(
    "/kaggle/working/benchmark_outputs",
    message=f"Phase 0A complete â€” {manifest.records} records"
)

# ============================================================
# SUMMARY
# ============================================================
print("\n" + "=" * 50)
print("OPENQSIM PHASE 0A SWEEP COMPLETE")
print("=" * 50)
print(f"  Total records:    {manifest.records}")
print(f"  Successful runs:  {manifest.successful_runs}")
print(f"  OOM failures:     {manifest.oom_failures}")
print(f"  Unique circuits:  {manifest.unique_circuits}")
print(f"  Backends:         {manifest.backends}")
print(f"  Qubit range:      {manifest.qubit_range}")
print(f"  Depth range:      {manifest.depth_range}")
print(f"  Dataset:          lekkalaharsha/openqsim-benchmarks")
print("=" * 50)